# Individual and Market Demand: Perfect Substitutes

Six consumers have **different perfect-substitutes utility functions** and incomes:

\[U_i(x,y)=a_i x+b_i y,\qquad p_x x+p_y y\le m_i.\]

We vary the **price of good X** on the vertical axis and plot **quantity of X** on the horizontal axis. All consumers face the same prices, but have different tastes and incomes.

For positive prices and incomes:

\[x_i^*(p_x)=\begin{cases}m_i/p_x&\text{if }a_i/p_x>b_i/p_y,\\0&\text{if }a_i/p_x<b_i/p_y.\end{cases}\]

At equality, *every affordable mix is optimal*, so X demand is a correspondence, not a unique number. In the code, `tie_rule='x'` assigns all spending to X at equality for plotting; set `'y'` to assign none. The jump at the threshold is plotted as disconnected pieces.

Market demand is the **horizontal sum**: \(X^M(p_x)=\sum_{i=1}^6 x_i^*(p_x)\).

In [ ]:
# Install once if needed (uncomment):
# %pip install ipywidgets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# In classic Jupyter notebook, the default inline backend works well.
# For live Matplotlib figure manipulation, optionally install ipympl and use %matplotlib widget.

In [ ]:
# EDIT THESE SIX CONSUMERS to fit your lecture.
# Utility is U_i(X,Y) = a_i * X + b_i * Y.
consumers = [
    {"name": "Consumer 1", "a": 1.0, "b": 1.0, "income": 40},
    {"name": "Consumer 2", "a": 1.5, "b": 1.0, "income": 60},
    {"name": "Consumer 3", "a": 2.0, "b": 1.0, "income": 80},
    {"name": "Consumer 4", "a": 2.5, "b": 1.0, "income": 50},
    {"name": "Consumer 5", "a": 3.0, "b": 1.0, "income": 100},
    {"name": "Consumer 6", "a": 3.5, "b": 1.0, "income": 70},
]

# Price range of X to plot (strictly positive), and initial price of Y.
PX_MIN, PX_MAX = 0.25, 6.0
INITIAL_PY = 1.0
TIE_RULE = "x"  # "x" or "y"; ties admit multiple optimal bundles.

In [ ]:
def x_demand(px, py, income, a, b, tie_rule="x"):
    """Demand for X for utility a*X+b*Y and budget px*X+py*Y<=income.

    px may be a scalar or NumPy array. Positive px,py,a,b required.
    """
    px = np.asarray(px, dtype=float)
    if py <= 0 or np.any(px <= 0) or a <= 0 or b <= 0:
        raise ValueError("Prices and utility weights must be positive.")
    threshold = a * py / b
    buys_x = (px < threshold) | (np.isclose(px, threshold, atol=1e-10, rtol=0) & (tie_rule == "x"))
    return np.where(buys_x, income / px, 0.0)


def market_demand(px, py, incomes):
    return sum(x_demand(px, py, m, c["a"], c["b"], TIE_RULE)
               for c, m in zip(consumers, incomes))


def draw_demand(py, px_current, **income_values):
    incomes = [income_values[f"income_{i}"] for i in range(6)]
    px_grid = np.linspace(PX_MIN, PX_MAX, 2000)

    fig = plt.figure(figsize=(14, 11), constrained_layout=True)
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1.35])
    axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(3)]
    ax_market = fig.add_subplot(gs[2, :])

    # Plot each smooth segment separately, avoiding an artificial line across jumps.
    # For inverse-demand-style axes: horizontal = Qx, vertical = Px.
    for i, (ax, c, m) in enumerate(zip(axes, consumers, incomes)):
        threshold = c["a"] * py / c["b"]
        active = px_grid < threshold
        inactive = px_grid > threshold
        ax.plot(m / px_grid[active], px_grid[active], lw=2, color=f"C{i}")
        if inactive.any():
            ax.plot(np.zeros(inactive.sum()), px_grid[inactive], lw=2, color=f"C{i}")
        q_now = float(x_demand(px_current, py, m, c["a"], c["b"], TIE_RULE))
        ax.scatter([q_now], [px_current], color="black", zorder=4, s=30)
        ax.axhline(px_current, ls=":", color="gray", lw=0.9)
        ax.set_title(f"{c['name']}: U={c['a']:g}X+{c['b']:g}Y, m=${m:g}", fontsize=10)
        ax.set_xlabel("Quantity of X")
        ax.set_ylabel("Price of X")
        ax.set_ylim(PX_MIN, PX_MAX)
        ax.set_xlim(left=0)
        ax.grid(alpha=0.2)

    # Market demand is piecewise; split the curve at all switching prices.
    breaks = sorted({PX_MIN, PX_MAX} | {
        c["a"] * py / c["b"] for c in consumers
        if PX_MIN < c["a"] * py / c["b"] < PX_MAX
    })
    for low, high in zip(breaks[:-1], breaks[1:]):
        segment_px = np.linspace(low, high, 300)[1:-1]  # exclude ties/endpoints
        segment_q = market_demand(segment_px, py, incomes)
        ax_market.plot(segment_q, segment_px, color="tab:blue", lw=2.5)

    q_market_now = float(market_demand(px_current, py, incomes))
    ax_market.scatter([q_market_now], [px_current], color="black", zorder=5, s=60,
                      label=f"At Px=${px_current:.2f}: Market Qx={q_market_now:.1f}")
    ax_market.axhline(px_current, color="gray", linestyle=":", lw=1)
    ax_market.set_title("MARKET DEMAND = horizontal sum of six individual demands", fontsize=13)
    ax_market.set_xlabel("Total quantity of X")
    ax_market.set_ylabel("Price of X")
    ax_market.set_ylim(PX_MIN, PX_MAX)
    ax_market.set_xlim(left=0)
    ax_market.grid(alpha=0.2)
    ax_market.legend(loc="upper right")

    plt.show()

    print(f"At Px=${px_current:.2f} and Py=${py:.2f}:")
    print("Individual X demand:", [round(float(x_demand(px_current, py, m, c['a'], c['b'], TIE_RULE)), 2)
                                  for c, m in zip(consumers, incomes)])
    print(f"Market X demand: {q_market_now:.2f}")

In [ ]:
# Interactive controls: drag prices or change consumers' incomes.
style = {"description_width": "115px"}
price_y_slider = widgets.FloatSlider(value=INITIAL_PY, min=0.5, max=3.0, step=0.1,
                                     description="Price of Y", style=style,
                                     continuous_update=False)
price_x_slider = widgets.FloatSlider(value=1.5, min=PX_MIN, max=PX_MAX, step=0.05,
                                     description="Current Px", style=style,
                                     continuous_update=False)
income_sliders = {
    f"income_{i}": widgets.IntSlider(value=c["income"], min=10, max=200, step=10,
                                      description=f"Consumer {i + 1} income", style=style,
                                      continuous_update=False)
    for i, c in enumerate(consumers)
}

controls = {"py": price_y_slider, "px_current": price_x_slider, **income_sliders}
output = widgets.interactive_output(draw_demand, controls)
display(widgets.VBox([
    widgets.HTML("<b>Prices</b>"), widgets.HBox([price_y_slider, price_x_slider]),
    widgets.HTML("<b>Consumer incomes</b>"),
    widgets.GridBox(list(income_sliders.values()),
                    layout=widgets.Layout(grid_template_columns="repeat(2, minmax(300px, 1fr))")),
    output,
]))

## Ways to adapt this skeleton

- Edit the six `a`, `b`, and `income` entries in `consumers`. After changing the list, rerun the control and plot cells.
- The switching price for each consumer is \(p_x^*=a_i p_y/b_i\). Below that price the consumer spends all income on X; above it the consumer purchases no X.
- Sliders currently adjust **incomes and both prices**; add `a_i` and `b_i` sliders if students should experiment with tastes.
- Because this is a *perfect-substitutes* example, demand can jump when prices cross switching points. The individual demand at an exact tie is **set-valued**; the black marker uses the chosen `TIE_RULE` convention.
- To embed directly in a static HTML page you would need a different approach; this notebook needs a running Jupyter kernel for live widgets.